In [ ]:
import pandas as pd
import numpy as np

# Read in Population Data
pop = pd.read_excel('NGA_LGA_Population_2020.xlsx', sheet_name='POP_2020')

print(pop.head(10))

print(pop['SenDist_en'].nunique())
print(pop['ADM2_NAME'].nunique())

# Rename T_TL to POP_2020 for clarity
pop.rename(columns={'T_TL': 'POP_2020'}, inplace=True)

# Read in PGR Projection Data
pgr = pd.read_excel('PGR_PROJ_STATE.xlsx', sheet_name='PGR_Proj')

# Drop PGR_PROJ_2025 column from pgr dataframe since we will calculate it ourselves
pgr.drop(columns=['PGR_PROJ_2025'], inplace=True)

# Nasarawa State is missing from the PGR projection data, so we will add a row for Nasarawa in this dataframe and fill PGR_BOTH_2022 as 2.8 and PGR_PROJ_2025 as (1+(PGR_BOTH_2022/100))^3
# Source: https://citypopulation.de/en/nigeria/admin/16__nasarawa/
nasarawa_row = pd.DataFrame({
    'ADM1_NAME': ['Nasarawa'],
    'PGR_BOTH_2022': [2.8]
})
pgr = pd.concat([pgr, nasarawa_row], ignore_index=True)

# Calculate PGR_PROJ_2025 for all states using the formula: (1 + (PGR_BOTH_2022 / 100))^3
pgr['PGR_PROJ_2025'] = (1 + (pgr['PGR_BOTH_2022'] / 100)) ** 3

print(pop.columns)
print(pgr.columns)

  ADM0_NAME ADM1_NAME          ADM2_NAME    SenDist_en    T_TL
0   NIGERIA      ABIA          ABA NORTH    Abia South  115305
1   NIGERIA      ABIA          ABA SOUTH    Abia South  384940
2   NIGERIA      ABIA          AROCHUKWU    Abia North  255439
3   NIGERIA      ABIA              BENDE    Abia North  244495
4   NIGERIA      ABIA            IKWUANO  Abia Central  316803
5   NIGERIA      ABIA  Isiala-Ngwa North  Abia Central  221635
6   NIGERIA      ABIA  Isiala-Ngwa South  Abia Central  165135
7   NIGERIA      ABIA         Isiukwuato    Abia North  181063
8   NIGERIA      ABIA           Obi Ngwa    Abia South  238027
9   NIGERIA      ABIA             OHAFIA    Abia North  353882
109
771
Index(['ADM0_NAME', 'ADM1_NAME', 'ADM2_NAME', 'SenDist_en', 'POP_2020'], dtype='object')
Index(['ADM1_NAME', 'PGR_BOTH_2022', 'PGR_PROJ_2025'], dtype='object')


In [54]:
"""
I will merge the population data with the PGR projection data. The PGR data is at the state level and the population data is at the LGA level.

First, I will need to ensure that the state names in both datasets match.
"""

# Check unique state names in population data
print("Unique state names in population data:")
print(pop['ADM1_NAME'].unique())

# Check unique state names in PGR data
print("\nUnique state names in PGR data:")
print(pgr['ADM1_NAME'].unique())

# It looks like the state names match in both datasets, just that state names in 'pop' are in uppercase. I will convert the state names in 'pgr' to uppercase to ensure they match.
pgr['ADM1_NAME'] = pgr['ADM1_NAME'].str.upper()

# I will drop the row with 'TOTAL' in the PGR data as it is not needed for the merge.
pgr = pgr[pgr['ADM1_NAME'] != 'TOTAL']

# Merge the population data with the PGR projection data on the state name (ADM1_NAME)
pop_pgr = pd.merge(pop, pgr, on='ADM1_NAME', how='left')


Unique state names in population data:
['ABIA' 'ADAMAWA' 'AKWA IBOM' 'ANAMBRA' 'BAUCHI' 'BAYELSA' 'BENUE' 'BORNO'
 'CROSS RIVER' 'DELTA' 'EBONYI' 'EDO' 'EKITI' 'ENUGU'
 'FEDERAL CAPITAL TERRITORY' 'GOMBE' 'IMO' 'JIGAWA' 'KADUNA' 'KANO'
 'KATSINA' 'KEBBI' 'KOGI' 'KWARA' 'LAGOS' 'NASARAWA' 'NIGER' 'OGUN' 'ONDO'
 'OSUN' 'OYO' 'PLATEAU' 'RIVERS' 'SOKOTO' 'TARABA' 'YOBE' 'ZAMFARA']

Unique state names in PGR data:
['Abia' 'Adamawa' 'Akwa Ibom' 'Anambra' 'Bauchi' 'Bayelsa' 'Benue' 'Borno'
 'Cross River' 'Delta' 'Ebonyi' 'Edo' 'Ekiti' 'Enugu'
 'Federal Capital Territory' 'Gombe' 'Imo' 'Jigawa' 'Kaduna' 'Kano'
 'Katsina' 'Kebbi' 'Kogi' 'Kwara' 'Lagos' 'Niger' 'Ogun' 'Ondo' 'Osun'
 'Oyo' 'Plateau' 'Rivers' 'Sokoto' 'Taraba' 'Yobe' 'Zamfara' 'Total'
 'Nasarawa']


In [55]:
print(pop_pgr.head(10))

  ADM0_NAME ADM1_NAME          ADM2_NAME    SenDist_en  POP_2020  \
0   NIGERIA      ABIA          ABA NORTH    Abia South    115305   
1   NIGERIA      ABIA          ABA SOUTH    Abia South    384940   
2   NIGERIA      ABIA          AROCHUKWU    Abia North    255439   
3   NIGERIA      ABIA              BENDE    Abia North    244495   
4   NIGERIA      ABIA            IKWUANO  Abia Central    316803   
5   NIGERIA      ABIA  Isiala-Ngwa North  Abia Central    221635   
6   NIGERIA      ABIA  Isiala-Ngwa South  Abia Central    165135   
7   NIGERIA      ABIA         Isiukwuato    Abia North    181063   
8   NIGERIA      ABIA           Obi Ngwa    Abia South    238027   
9   NIGERIA      ABIA             OHAFIA    Abia North    353882   

   PGR_BOTH_2022  PGR_PROJ_2025  
0           2.35            NaN  
1           2.35            NaN  
2           2.35            NaN  
3           2.35            NaN  
4           2.35            NaN  
5           2.35            NaN  
6           2

In [56]:
pop_pgr['POP_PROJ_2025'] = pop_pgr['POP_2020'] * pop_pgr['PGR_PROJ_2025']
print(pop_pgr[['ADM1_NAME', 'ADM2_NAME', 'POP_2020', 'POP_PROJ_2025']].head(10))

pop_pgr.to_excel('NGA_LGA_Population_PGR_Projections.xlsx', index=False)

  ADM1_NAME          ADM2_NAME  POP_2020  POP_PROJ_2025
0      ABIA          ABA NORTH    115305            NaN
1      ABIA          ABA SOUTH    384940            NaN
2      ABIA          AROCHUKWU    255439            NaN
3      ABIA              BENDE    244495            NaN
4      ABIA            IKWUANO    316803            NaN
5      ABIA  Isiala-Ngwa North    221635            NaN
6      ABIA  Isiala-Ngwa South    165135            NaN
7      ABIA         Isiukwuato    181063            NaN
8      ABIA           Obi Ngwa    238027            NaN
9      ABIA             OHAFIA    353882            NaN
